In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')

    GIT_BRANCH = 'refactor-continuo'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import os
import glob
import json
import pandas as pd
import rasterio
from rasterio.warp import transform
import rioxarray
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm

path_ground_truth = DATA_DIR / "interim" / "points.json"
path_sentinel2_data = DATA_DIR / "processed" / "sentinel2_data"
path_legend = DATA_DIR / "processed" / "legend.json"

df_ground_truth = pd.read_json(path_ground_truth)

# Esclusione classi fittizie 'Undecided' di Copernicus (3100 e 3200)
UNDECIDED_CLASSES = [3100, 3200]
initial_count = len(df_ground_truth)
df_ground_truth = df_ground_truth[~df_ground_truth['code'].astype(int).isin(UNDECIDED_CLASSES)].reset_index(drop=True)
print(f"Filtrati {initial_count - len(df_ground_truth)} punti appartenenti alle classi 'Undecided' ({UNDECIDED_CLASSES}). Campi da elaborare: {len(df_ground_truth)}")

legend = {}
if path_legend.exists():
    with open(path_legend, "r", encoding="utf-8") as f:
        legend = json.load(f)

def process_single_point(args):
    index, row_dict, base_data_dir, legend_map = args
    crop_id = int(row_dict["code"])
    if crop_id in [3100, 3200]:
        return index, None

    lon_val = float(row_dict["lon"])
    lat_val = float(row_dict["lat"])

    point_dir = base_data_dir / f"point_{index}"
    tif_files = list(point_dir.glob("sentinel2_data_*.tif"))
    if not tif_files:
        return index, None

    point_tif = tif_files[0]
    try:
        with rasterio.open(point_tif) as src:
            data = src.read()  # shape: (n_layers, height, width)
            n_layers = src.count
            if n_layers < 6:
                return index, None

            # Conversione coordinate GPS nel CRS del raster Sentinel-2
            xs, ys = transform("EPSG:4326", src.crs, [lon_val], [lat_val])
            r_center, c_center = src.index(xs[0], ys[0])

            # Controllo limiti immagine
            if r_center < 0 or r_center >= data.shape[1] or c_center < 0 or c_center >= data.shape[2]:
                return index, None

            # Finestra 3x3 pixel (30m x 30m) centrata sul campo
            r_min = max(0, r_center - 1)
            r_max = min(data.shape[1], r_center + 2)
            c_min = max(0, c_center - 1)
            c_max = min(data.shape[2], c_center + 2)

            crop_patch = data[:, r_min:r_max, c_min:c_max]

            # Valore mediano per ciascuno strato della finestra 3x3 sui pixel validi
            layer_values = []
            for l in range(n_layers):
                valid_vals = crop_patch[l][crop_patch[l] > 0]
                layer_values.append(float(np.median(valid_vals)) if len(valid_vals) > 0 else 0.0)

            n_months = min(12, n_layers // 6)

            # Medie annuali delle 6 bande calcolate sulla finestra 3x3
            def extract_clean_band_mean(b_offset):
                vals = [layer_values[l] for l in range(b_offset, n_layers, 6) if layer_values[l] > 0]
                return float(np.mean(vals)) if vals else 0.0

            record = {
                "ID_Campo": index,
                "Ground_Truth": crop_id,
                "Crop_Name": legend_map.get(str(crop_id), str(crop_id)),
                "Blu_B02": extract_clean_band_mean(0),
                "Verde_B03": extract_clean_band_mean(1),
                "Rosso_B04": extract_clean_band_mean(2),
                "NIR_B08": extract_clean_band_mean(3),
                "SWIR1_B11": extract_clean_band_mean(4),
                "SWIR2_B12": extract_clean_band_mean(5),
            }

            monthly_ndvis = []
            monthly_ndwis = []

            # Serie temporali mensili
            for m in range(12):
                month_str = f"{m+1:02d}"
                if m < n_months:
                    red_v = layer_values[m * 6 + 2]
                    nir_v = layer_values[m * 6 + 3]
                    swir1_v = layer_values[m * 6 + 4]

                    # Controllo rigoroso: entrambe le bande devono essere > 0 (esclude nodata)
                    ndvi_m = (nir_v - red_v) / (nir_v + red_v) if (nir_v > 0 and red_v > 0) else 0.0
                    ndwi_m = (nir_v - swir1_v) / (nir_v + swir1_v) if (nir_v > 0 and swir1_v > 0) else 0.0
                else:
                    ndvi_m, ndwi_m = 0.0, 0.0

                monthly_ndvis.append(ndvi_m)
                monthly_ndwis.append(ndwi_m)
                record[f"NDVI_{month_str}"] = round(ndvi_m, 4)
                record[f"NDWI_{month_str}"] = round(ndwi_m, 4)

            # Indicatori fenologici
            valid_ndvis = [v for v in monthly_ndvis if v > 0]
            if valid_ndvis:
                record["NDVI_max"] = round(max(valid_ndvis), 4)
                record["NDVI_min"] = round(min(valid_ndvis), 4)
                record["NDVI_amp"] = round(record["NDVI_max"] - record["NDVI_min"], 4)
                record["Peak_Month"] = int(np.argmax(monthly_ndvis) + 1)
            else:
                record["NDVI_max"] = 0.0
                record["NDVI_min"] = 0.0
                record["NDVI_amp"] = 0.0
                record["Peak_Month"] = 0

            return index, record
    except Exception as e:
        return index, None

# Preparazione ed esecuzione task in parallelo con ThreadPoolExecutor
tasks = [(index, row.to_dict(), path_sentinel2_data, legend) for index, row in df_ground_truth.iterrows()]
results = [None] * len(tasks)
workers = min(16, os.cpu_count() or 4)

print(f"Avvio estrazione serie temporali su {len(tasks)} campi con {workers} worker...")

with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(process_single_point, t) for t in tasks]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Elaborazione campi"):
        idx, res = future.result()
        if res is not None:
            results[idx] = res

# Creazione del dataframe finale
valid_results = [r for r in results if r is not None]
final_df = pd.DataFrame(valid_results)
print(f"Estrazione completata! Dataset creato con {len(final_df)} campi validi.")

In [ ]:
dataset_dir = DATA_DIR / 'processed' / 'dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

# salva il dataset in formato Parquet
parquet_path = dataset_dir / 'dataset.parquet'
final_df.to_parquet(parquet_path, index=False)

# salva il dataset in formato CSV
csv_path = dataset_dir / 'dataset.csv'
final_df.to_csv(csv_path, index=False)

print(f"Dataset salvati con successo nella cartella: {dataset_dir.resolve()}")       
print(f"\nDimensioni tabella: {final_df.shape[0]} righe x {final_df.shape[1]} colonne\n")

# mostra le prime 5 righe del dataset
display(final_df.head())